## Why we need *input → target* pairs

During pre-training an LLM is shown a short **context** (the *input*) and asked to guess the very next token (the *target*).
If the corpus is already tokenised into the integer IDs that the model understands, the most direct way to create those pairs is to keep two sliding copies of the same sequence, offset by one position:

```
Sequence:   [t_0, t_1, t_2, t_3, ..., t_n]
Input  x:   [t_0, t_1, t_2, t_3, ..., t_{n-1}]
Target y:       [t_1, t_2, t_3, ..., t_n]
```

Every step of the window teaches the model *“given x\[i] predict y\[i]”*.

---

In [ ]:
# !pip install transformers torch --upgrade

In [6]:
# -------------------------
# Tokenize the corpus
# -------------------------
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset, DataLoader
from nltk.corpus import gutenberg

sample_text = gutenberg.raw('austen-emma.txt')[:80_000]

# a BPE-based tokenizer (GPT-2’s is convenient and already trained)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

token_ids = tokenizer.encode(sample_text, add_special_tokens=False)
print(f"Total tokens after BPE: {len(token_ids):,}")

# ---------------------------------------------
# Show the simplest x / y construction
# ---------------------------------------------
x_ids = token_ids[:-1]
y_ids = token_ids[1:]

# Show ten (id → id) pairs …
for i in range(10):
    print(f"{x_ids[i]:>6}  →  {y_ids[i]}")

# … and the same pairs back-translated to text.
for i in range(10):
    print(f'{tokenizer.decode([x_ids[i]])!r:>12}  →  {tokenizer.decode([y_ids[i]])!r}')

Token indices sequence length is longer than the specified maximum sequence length for this model (19884 > 1024). Running this sequence through the model will result in indexing errors


Total tokens after BPE: 19,884
    58  →  10161
 10161  →  2611
  2611  →  416
   416  →  12091
 12091  →  2517
  2517  →  268
   268  →  1248
  1248  →  1433
  1433  →  60
    60  →  198
         '['  →  'Em'
        'Em'  →  'ma'
        'ma'  →  ' by'
       ' by'  →  ' Jane'
     ' Jane'  →  ' Aust'
     ' Aust'  →  'en'
        'en'  →  ' 18'
       ' 18'  →  '16'
        '16'  →  ']'
         ']'  →  '\n'


You should see output in the style

```
  58 → 10161
'[' → 'Em'
```

confirming that every **input token** is aligned with the **target token** one step to the right.

---

In [8]:
from typing import Sequence
import torch
from torch.utils.data import Dataset, DataLoader

# -----------------------------------------------------------
# An efficient PyTorch Dataset using a sliding window
# -----------------------------------------------------------

class SlidingWindowDataset(Dataset):
    """
    Return (input_window, target_window) pairs for next-token prediction.
    
    Parameters
    ----------
    ids : Sequence[int]
        Pre-tokenised corpus as integer IDs.
    window_size : int >= 1
        Number of tokens shown to the model.
    stride : int >= 1
        Step by which the window is advanced.
    """
    def __init__(self, ids: Sequence[int], window_size: int, stride: int = 1):
        if window_size < 1 or stride < 1:
            raise ValueError("`window_size` and `stride` must be ≥ 1")
        self.ids = ids
        self.window_size = window_size
        self.stride = stride
        self.last_start = len(ids) - window_size - 1        # final legal start idx

    def __len__(self) -> int:
        # ⌊(N − window_size)/stride⌋ + 1 windows in total
        return 1 + (self.last_start // self.stride)

    def __getitem__(self, idx: int):
        start = idx * self.stride
        x = self.ids[start            : start + self.window_size    ]
        y = self.ids[start + 1        : start + self.window_size + 1]
        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long),
        )

# ------------------------------------------
# Wrap it in a standard DataLoader
# ------------------------------------------
WINDOW = 128          # context length shown to the model
STRIDE = 1            # 1 → fully overlapping windows; larger = subsampling
BATCH  = 32           # tune to fit GPU memory

dataset = SlidingWindowDataset(token_ids, window_size=WINDOW, stride=STRIDE)
loader  = DataLoader(dataset, batch_size=BATCH, shuffle=True, drop_last=True)

# Quick sanity check: fetch one batch and inspect its shape.
x_batch, y_batch = next(iter(loader))
print("batch   :", x_batch.shape)   # (BATCH, WINDOW)
print("targets :", y_batch.shape)   # identical

batch   : torch.Size([32, 128])
targets : torch.Size([32, 128])


The data loader now hands the training loop two tensors per step:

* **`x_batch`** – the context the LLM will read
* **`y_batch`** – the next-token ground truth the loss will compare against

Both are plain 2-D `LongTensor`s ready for an embedding layer.

---

## Step-by-step walk-through

1. **Tokenization**
   We rely on a Byte-Pair-Encoding (BPE) vocabulary (`gpt2`’s for convenience).
   Tokenization produces a *single vector of IDs* – no padding, no batching yet.

2. **Offset pair construction**
   Two slices of that vector (`[:-1]` and `[1:]`) create perfectly aligned
   `(input, target)` arrays where each `target[i]` is *exactly* the token that follows `input[i]` in the original text.

3. **SlidingWindowDataset**

   * `window_size` controls the context length fed to the model (also known as "Block size").
   * `stride` lets you choose how much overlap successive windows share (i.e., how many tokens you move forward to generate the next sample).
     Setting it to `window_size` yields non-overlapping chunks; smaller values give denser coverage at the cost of more samples. 

4. **PyTorch collation**
   In PyTorch, **collation** refers to the process of combining a list of samples into a single batch when using a DataLoader. When you use a `DataLoader`, it retrieves several samples from the dataset and collates them into a batch. Without special treatment, the default collate function stacks the returned tensors along a new first dimension, yielding shape `(batch, window_size)`, exactly what most transformer models expect.

5. **Integration with a training loop**
   Each iteration of

   ```python
   for x, y in loader:
       logits = model(x)            # (batch, window, vocab)
       loss   = criterion(logits.flatten(0,1), y.flatten(0,1))
       ...
   ```

   computes a categorical cross-entropy between the model’s prediction at every position and the sliding target, teaching "next-token" behavior.

---

> In the context of generating input-target pairs for LLM training:
> 
> * **Window size (`block_size`)** is how many tokens you take as input for each sample.  
> * **Stride** is how many tokens you move forward to generate the next sample.
> 
> ---
> 
> ## **Case 1: stride = 1**
> 
> This is the **most overlapping** case. You move one token at a time.
> 
> ```python
> tokens = [10, 11, 12, 13, 14, 15, 16, 17]
> block_size = 4
> stride = 1
> 
> Samples:
> x1 = [10, 11, 12, 13]   y1 = [11, 12, 13, 14]
> x2 = [11, 12, 13, 14]   y2 = [12, 13, 14, 15]
> x3 = [12, 13, 14, 15]   y3 = [13, 14, 15, 16]
> ...
> ```
> 
> This generates **many overlapping samples**. It is **ideal for maximum data usage**, but more computationally expensive.
> 
> ---
> 
> ## **Case 2: stride = block_size**
> 
> This is the **non-overlapping** case.
> 
> ```python
> tokens = [10, 11, 12, 13, 14, 15, 16, 17]
> block_size = 4
> stride = 4
> 
> Samples:
> x1 = [10, 11, 12, 13]   y1 = [11, 12, 13, 14]
> x2 = [14, 15, 16, 17]   y2 = [15, 16, 17, ??]  ← not enough tokens for next prediction, may need padding or truncation
> ```
> 
> This generates **fewer samples**, each non-overlapping.
> 
> ---
> 
> ## **Case 3: stride < block_size**
> 
> This is a **partial overlap** case:
> 
> ```python
> block_size = 4
> stride = 2
> 
> x1 = [10, 11, 12, 13]   y1 = [11, 12, 13, 14]
> x2 = [12, 13, 14, 15]   y2 = [13, 14, 15, 16]
> ```
> 
> Here, each new window starts 2 tokens after the previous, so the windows **overlap partially**.
> 
> ---
> 
> ## **When to use which?**
> 
> * `stride=1`: more data, better for smaller datasets, but computationally expensive.  
> * `stride=block_size`: faster, fewer samples, less redundancy.  
> * `stride < block_size`: compromise between redundancy and efficiency.


---

**Next Step**

* **Dynamic masking** – during fine-tuning you might mask future tokens beyond the target position (`torch.triu` on the attention mask) rather than physically removing them in the dataset.
* **Packing multiple documents** – pad shorter documents and add an *end-of-text* token so that prediction never leaks across document boundaries.
* **Streaming loaders** – for very large corpora use an iterable-style dataset that reads from disk on-the-fly instead of holding everything in memory.

This implementation, however, is all you need to feed an LLM with
correctly-aligned `(input, target)` tensors for classic next-token pre-training.

Next, we examine how a helper function can use a data loader and sample a batch of text. 

In [10]:
import tiktoken  # swap in your tokenizer of choice

def create_dataloader(
    text: str | Sequence[int],
    *,
    tokenizer: tiktoken.Encoding | None = None,
    batch_size: int = 32,
    window_size: int = 256,
    stride: int = 128,
    shuffle: bool = True,
    drop_last: bool = True,
    num_workers: int = 0,
):
    """
    Turn raw text *or* pre-tokenized IDs into a PyTorch DataLoader.
    """
    if isinstance(text, str):
        if tokenizer is None:
            tokenizer = tiktoken.get_encoding("gpt2")
        token_ids = tokenizer.encode(text)
    else:
        # already a list/array of ints
        token_ids = list(text)

    dataset = SlidingWindowDataset(
        token_ids, window_size=window_size, stride=stride
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return loader

> **Why a helper?**
> It centralizes the boilerplate—tokenization, dataset creation, and loader settings—so you can change stride, context length, or workers in one place.

Sanity-check with a toy corpus

In [11]:
toy = list(range(10))               # token IDs: [0, 1, …, 9]
loader = create_dataloader(
    toy,
    batch_size=1,
    window_size=4,
    stride=1,
    shuffle=False,                  # keep order for illustration
)

x_batch, y_batch = next(iter(loader))
print("inputs :", x_batch)          # tensor([[0, 1, 2, 3]])
print("targets:", y_batch)          # tensor([[1, 2, 3, 4]])

inputs : tensor([[0, 1, 2, 3]])
targets: tensor([[1, 2, 3, 4]])


*Output explained*

| index | `x` window | `y` window (shift + 1) |
| ----: | ---------- | ---------------------- |
|     0 | 0 1 2 3    | 1 2 3 4                |
|     1 | 1 2 3 4    | 2 3 4 5                |
|     … | …          | …                      |

With *window\_size = 4* and *stride = 1*, there are 6 sliding windows (length = 10, so 10 – 4 – 1 + 1 = 6). Larger strides sub-sample windows to reduce overlap (and GPU load) at the cost of fewer training examples.

> Why `drop_last=True`?
> 
> When the dataset length is not an exact multiple of *batch\_size*, PyTorch’s `DataLoader` would otherwise return a smaller final batch. For sequence models it is often cleaner to **discard** that remainder to keep every tensor in the batch rectangular.

Typical context lengths in cutting-edge LLMs (July 2025)

| Model (provider, release)                 | Maximum **context window** (tokens) | Citation           |
| ----------------------------------------- | ----------------------------------- | ------------------ |
| **GPT-4 Turbo** (OpenAI, Nov 2023)        | 128 000                             | ([OpenAI][1])      |
| **GPT-4.1** (OpenAI, Apr 2025)            | 1 000 000                           | ([OpenAI][2])      |
| **Claude 3** family (Anthropic, Mar 2024) | 200 000 (1 M limited preview)       | ([Anthropic][3])   |
| **Gemini 1.5 Pro** (Google, Feb 2024)     | 1 000 000 (experimental)            | ([blog.google][4]) |
| **Mistral Large** (Mistral AI, Dec 2023)  | 32 000                              | ([Mistral AI][5])  |


> **Rule of thumb**
> Research-grade transformers today rarely train with fewer than 256–2 048 tokens. Longer windows (32 k – 1 M) are feasible but require efficient attention mechanisms (e.g., flash attention 2, sliding-window attention, or chunked RoPE) and **significantly** more VRAM.

---

[1]: https://openai.com/index/new-models-and-developer-products-announced-at-devday/?utm_source=chatgpt.com "New models and developer products announced at DevDay - OpenAI"
[2]: https://openai.com/index/gpt-4-1/?utm_source=chatgpt.com "Introducing GPT-4.1 in the API - OpenAI"
[3]: https://www.anthropic.com/news/claude-3-family?utm_source=chatgpt.com "Introducing the next generation of Claude - Anthropic"
[4]: https://blog.google/technology/ai/google-gemini-next-generation-model-february-2024/?utm_source=chatgpt.com "Our next-generation model: Gemini 1.5 - Google Blog"
[5]: https://mistral.ai/news/mistral-large?utm_source=chatgpt.com "Au Large | Mistral AI"

Let’s now explore what happens when we sample multiple sequences per training batch using a `batch_size > 1`.

To make the behavior crystal clear, we can use a small dataset, a minimal context length of 2, and a stride of 2:

In [12]:
# Simulated corpus (token IDs)
raw_text = list(range(10))   # Example: token IDs = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# Initialize the dataloader with a larger batch size
loader = create_dataloader(
    raw_text,
    batch_size=4,             # batch of 4 (ideal to illustrate window independence)
    window_size=2,            # short context length (just 2 tokens)
    stride=2,                 # step size of 2 (non-overlapping windows)
    shuffle=False             # deterministic for reproducibility
)

# Convert dataloader to an iterator and fetch the first batch
data_iter = iter(loader)
inputs, targets = next(data_iter)

# Visualize the result
print("Inputs:\n", inputs)
print("Targets:\n", targets)

Inputs:
 tensor([[0, 1],
        [2, 3],
        [4, 5],
        [6, 7]])
Targets:
 tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]])


Suppose `raw_text = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]`.

Then, given `window_size=2` and `stride=2`, the `SlidingWindowDataset` constructs the following sequences:

| Example | Input Tokens (x) | Target Tokens (y) |
| ------- | ---------------- | ----------------- |
| 0       | `[0, 1]`         | `[1, 2]`          |
| 1       | `[2, 3]`         | `[3, 4]`          |
| 2       | `[4, 5]`         | `[5, 6]`          |
| 3       | `[6, 7]`         | `[7, 8]`          |

Since `batch_size=4`, we receive these four sequences all at once. Each *row* of the output tensor is an independent training sample.


| Stride | Overlap | Description                                                                                                |
| ------ | ------- | ---------------------------------------------------------------------------------------------------------- |
| `1`    | High    | **Maximally overlapping** windows, increases training examples but can lead to redundancy and overfitting. |
| `2`    | None    | **Non-overlapping** windows of `window_size=2` (as used above).                                            |
| `>2`   | Sparse  | **Under-sampling** the token stream—trading sample diversity for speed and lower memory usage.             |

Setting `stride=2` ensures each window is completely **non-overlapping**, which gives us a *clean, disjoint partition* of the input space. This can reduce **redundancy**, lower **overfitting risk**, and be useful for regularization during training. However, it does **reduce the total number of training samples**, which may affect convergence speed.

* A **larger batch size** allows the model to process more examples in parallel, which is critical for GPU acceleration.
* **Stride > 1** reduces redundancy between windows—especially valuable when training on long corpora.
* **Shuffling** is often disabled in toy examples for deterministic behavior, but should be enabled (`shuffle=True`) during real training to prevent learning order-specific artifacts.
* The combination of `window_size`, `stride`, and `batch_size` directly shapes the **training signal** seen by the model.
